# 02 - Benchmark of disentanglement strategies

All numbers are means over cross-validation folds of held-out metrics. Variants:

| variant | latent | regularisers |
|---|---|---|
| baseline | one 64-d space | MOSA as released (conditional decoders) |
| adversarial | one 64-d space | MOSA source discriminator |
| dip | one 64-d space | DIP-VAE-II on the aggregate posterior |
| supervised | blocks (source 4, tissue 12, EMT 2, free 48) | supervised heads |
| supervised_nocond | blocks | heads + source adversary, decoders do not see the source |
| hsic | blocks | heads + CKA independence (Lopez et al. 2018) |
| midas | blocks | heads + leakage discriminators (MIDAS-style) |
| full | blocks | heads + discriminators + CKA |
| full_nocond | blocks | as full, confusion-mode adversaries, decoders blind to source |
| sweep_adv*_hsic* | blocks | full_nocond with stronger source penalties (sim_c05 only) |

In [ ]:
# parameters
RESULTS_DIR = "results"
SIM_DATASETS = ["sim_c00", "sim_c05", "sim_c09"]
REAL_DATASET = "proteomics_paired"

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from mosa_disentangle import plotting
from mosa_disentangle.data import read_dataset
from mosa_disentangle.training import block_columns

sns.set_context("paper")
plt.rcParams["figure.dpi"] = 110

In [ ]:
tables = Path(RESULTS_DIR) / "tables"
long = pd.read_csv(tables / "metrics_long.csv")
summary = pd.read_csv(tables / "metrics_summary.csv")
variants = [v for v in ["baseline", "adversarial", "adversarial_confusion", "dip", "supervised",
                        "supervised_nocond", "hsic", "midas", "full", "full_nocond"]
            if v in set(long["variant"])]
sweep = sorted(v for v in set(long["variant"]) if v.startswith("sweep_"))
sweep_summary = summary[summary["variant"].isin(sweep)].copy()
long = long[long["variant"].isin(variants)].copy()
summary = summary[summary["variant"].isin(variants)].copy()
long["variant"] = pd.Categorical(long["variant"], variants, ordered=True)
summary["variant"] = pd.Categorical(summary["variant"], variants, ordered=True)
summary = summary.sort_values(["dataset", "variant"])
long.groupby(["dataset", "variant"], observed=True).size().unstack()

## 1. Cost of disentanglement: held-out reconstruction

`nmse_mean` averages the per-view pooled NMSE (MOSA's score; 1 = predicting the mean). `impute_<view>` is the NMSE of a view predicted with that view hidden from the encoders (cross-view imputation, MOSA's main use case).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plotting.metric_bars(summary, "nmse_mean", ax=axes[0], datasets=SIM_DATASETS,
                     title="Held-out reconstruction NMSE (mean over views)")
plotting.metric_bars(summary, "nmse_proteomics", ax=axes[1], datasets=[REAL_DATASET],
                     title=f"{REAL_DATASET}: held-out NMSE")
plt.tight_layout()
cols = [c for c in summary.columns if c.startswith("impute_") and c.endswith("_mean")]
summary[summary["dataset"] == "sim_c05"].set_index("variant")[cols].round(3)

## 2. Does each block capture its own factor and nothing else?

Probe scores are held-out balanced accuracy (categorical) or R2 (continuous) of linear probes trained on a block. `*_own` is the chance-corrected score of the factor's own block, `*_leak` the best score among the other blocks (lower is better). `probe_all_*` is the predictability from the full latent space.

In [ ]:
own_leak = [c for c in summary.columns if (c.endswith("_own_mean") or c.endswith("_leak_mean"))]
view = summary[summary["dataset"].isin(SIM_DATASETS)].set_index(["dataset", "variant"])[own_leak]
view.columns = [c.replace("_mean", "") for c in view.columns]
view.round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ["probe_all_model_type", "model_type_leak", "tissue_own"]):
    plotting.metric_bars(summary, metric, ax=ax, datasets=SIM_DATASETS)
plt.tight_layout()

## 3. Mixing and biological structure in the biological blocks

`source_mixing_bio`: normalised kNN entropy of the source label in the biological blocks (1 = perfectly mixed). `tissue_silhouette_bio`: silhouette of tissue labels in the same blocks (higher = clearer tissue structure).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plotting.metric_bars(summary, "source_mixing_bio", ax=axes[0], datasets=SIM_DATASETS)
plotting.metric_bars(summary, "tissue_silhouette_bio", ax=axes[1], datasets=SIM_DATASETS)
plotting.metric_bars(summary, "source_mixing_bio", ax=axes[2], datasets=[REAL_DATASET],
                     title=f"{REAL_DATASET}: source mixing (bio blocks)")
plt.tight_layout()

## 4. Counterfactual batch removal against the ground truth

Validation samples of the non-reference pseudo-sources are decoded (i) as is, (ii) with the source conditional set to the reference (MOSA's mechanism) and (iii) with the conditional swapped **and** the source block replaced by the reference prototype. Scores are NMSE against the batch-free truth; `*_missing` scores cover views that were absent in the input (counterfactual imputation).

In [ ]:
cf_cols = [c for c in summary.columns if c.startswith("cf_") and c.endswith("_mean_mean")]
cf = summary[summary["dataset"].isin(SIM_DATASETS)].set_index(["dataset", "variant"])[cf_cols]
cf.columns = [c.replace("_mean_mean", "") for c in cf.columns]
cf.round(3)

In [ ]:
fig, axes = plt.subplots(1, len(SIM_DATASETS), figsize=(5 * len(SIM_DATASETS), 4), sharey=True)
for ax, ds in zip(np.atleast_1d(axes), SIM_DATASETS):
    sub = long[long["dataset"] == ds]
    melted = sub.melt(id_vars=["variant"], value_vars=[c for c in ["cf_none_mean", "cf_cond_mean", "cf_cond_block_mean"] if c in sub],
                      var_name="strategy", value_name="NMSE vs truth")
    melted["strategy"] = melted["strategy"].str.replace("cf_", "").str.replace("_mean", "")
    sns.barplot(data=melted, x="variant", y="NMSE vs truth", hue="strategy", ax=ax, errorbar="sd")
    ax.set_title(ds); ax.tick_params(axis="x", rotation=45)
plt.tight_layout()

## 4b. Real laboratories: paired cell lines

In `proteomics_paired`, 290 cell lines were measured by both Sanger and CCLE. `paired_top1_bio` is the fraction of CCLE rows whose Sanger measurement of the same line is the nearest Sanger neighbour in the biological blocks (chance ~ 1/300); `cf_cond_block_mean` is the NMSE of CCLE rows decoded as Sanger against the actual Sanger measurement (`cf_none_mean`: no correction).

In [ ]:
real = summary[summary["dataset"] == REAL_DATASET].set_index("variant")
cols = [c for c in ["paired_top1_bio_mean", "paired_top5_bio_mean", "paired_mrr_bio_mean", "paired_top1_all_mean",
                    "cf_none_mean_mean", "cf_cond_mean_mean", "cf_cond_block_mean_mean", "nmse_proteomics_mean",
                    "source_mixing_bio_mean", "tissue_silhouette_bio_mean"] if c in real.columns]
real[cols].rename(columns=lambda c: c.replace("_mean_mean", "").replace("_mean", "")).round(3)

## 5. Downstream informativeness

Mean held-out R2 of ridge models predicting the 200 most variable drug responses / CRISPR gene effects from the biological blocks.

In [ ]:
down = [c for c in summary.columns if c.startswith("downstream_") and c.endswith("_mean")]
summary[summary["dataset"].isin(SIM_DATASETS)].set_index(["dataset", "variant"])[down].round(3)

## 5b. Penalty-strength sweep (sim_c05)

Source `adversarial_weight` (confusion mode) and `hsic_weight` on top of `full_nocond`: what it costs in reconstruction to push the source out of the biological blocks. `diverged` is the fraction of folds whose optimisation produced non-finite latent codes (their other metrics are missing).

In [ ]:
sw = sweep_summary[sweep_summary["dataset"] == "sim_c05"].copy()
sw["adv"] = sw["variant"].astype(str).str.extract(r"adv([0-9p]+)")[0].str.replace("p", ".").astype(float)
sw["hsic"] = sw["variant"].astype(str).str.extract(r"hsic([0-9p]+)")[0].str.replace("p", ".").astype(float)
cols = ["diverged_mean", "nmse_mean_mean", "cf_cond_block_mean_mean", "model_type_leak_mean",
        "source_mixing_bio_mean", "tissue_own_mean", "downstream_drugresponse_bio_mean"]
fig, axes = plt.subplots(1, len(cols), figsize=(4 * len(cols), 3.5))
for ax, c in zip(axes, cols):
    sns.heatmap(sw.pivot(index="adv", columns="hsic", values=c), annot=True, fmt=".2f", cmap="viridis", ax=ax, cbar=False)
    ax.set_title(c.replace("_mean_mean", "").replace("_mean", ""))
plt.tight_layout()

## 6. Summary table

One row per variant on the moderately confounded simulation, ordered by the counterfactual score.

In [ ]:
keys = ["nmse_mean_mean", "cf_cond_block_mean_mean", "cf_cond_mean_mean", "model_type_leak_mean",
        "tissue_own_mean", "emt_own_mean", "source_mixing_bio_mean", "tissue_silhouette_bio_mean",
        "downstream_drugresponse_bio_mean", "fit_seconds_mean"]
keys = [k for k in keys if k in summary.columns]
best = summary[summary["dataset"] == "sim_c05"].set_index("variant")[keys]
best.columns = [c.replace("_mean_mean", "").replace("_mean", "") for c in best.columns]
best.round(3)